# CE541E08 — Unit 2 · Day 15 — while Loops: Reservoir Routing, Manning's n Iteration and Hardy-Cross
| | |
|---|---|
| **Course** | CE541E08 |
| **Department** | Civil Engineering · Christ University |
| **Instructor** | Dr. Arpan Pradhan |
| **Unit** | Unit 2 |
| **Session** | Day 15 of 45 |
| **CO** | CO2 |
| **Topics** | while loop · reservoir routing · back-calculate Manning's n · Hardy-Cross iteration |
---
> Read the explanation before each code block. Check the expected output. Run the cell and verify. Then try the small challenge.
---

In [ ]:
student_name = "Your Full Name"
roll_number  = "2024XXXXXX"
github_repo  = "https://github.com/your-username/CE541E08-2026"
session      = "Day 15"
print(f"CE541E08 | {student_name} | {roll_number} | {session}")

---
## Section 1 — The while Loop

A `while` loop repeats as long as a condition is True — used when the number of iterations is not known in advance:

```python
while condition:
    # body
```

Key engineering uses:
- **Reservoir routing** — simulate day by day until monsoon ends or reservoir fills
- **Iterative solvers** — repeat until a value converges (Manning's n calibration, Hardy-Cross)
- **Threshold detection** — keep scanning until an alarm condition is met

The danger with `while` loops is an infinite loop — always include a safety exit (maximum iteration counter).

---
## Code Block 1 — Reservoir Routing

### What this code does

We simulate a reservoir filling day by day during the monsoon season. The while loop advances one day at a time, adding inflow and subtracting outflow, stopping when the reservoir is full or the monsoon ends.

### Why each step is taken

**`while storage < capacity_Mm3 and day_index < len(inflow_Mm3day)`:**
Two conditions joined by `and`. The loop continues only when BOTH are True: reservoir not yet full AND inflow data still available. Either condition becoming False exits the loop.

**`day_index += 1`:**
Manual index increment. Unlike a for loop, the while loop does not automatically advance — you must do it yourself. Forgetting this causes an infinite loop.

**`if storage > capacity_Mm3`:**
After adding the day's net flow, storage might exceed capacity. This clips it to capacity and records the spill.

### Algorithm

```
1. capacity=50, storage=8, outflow=0.8
   inflow = 20-day monsoon sequence

2. while storage < 50 AND day_index < 20:
     Qin = inflow[day_index]
     storage = storage + Qin - outflow
     if storage > 50:
       spill = storage - 50
       storage = 50
     day_index += 1

3. After loop: print result
   if storage >= 50: "FULL on day X"
   else: "Monsoon ended, storage = Y"
```

### Expected output

```
 Day  Inflow  Outflow    Net  Storage  Note
────────────────────────────────────────────
   1    1.20     0.80   0.40     8.40
   2    1.80     0.80   1.00     9.40
   ...
  17    2.10     0.80   1.30    50.00  FULL
Reservoir FULL on Day 17. Total spill: X.XX Mm3
```

In [ ]:
# Level-pool reservoir routing
capacity_Mm3    = 50.0
initial_storage = 8.0
outflow_Mm3day  = 0.8   # constant controlled release

inflow_Mm3day = [1.2,1.8,3.4,6.7,12.3,18.5,22.1,19.8,15.4,
                 11.2,8.9,6.7,5.4,4.2,3.1,2.8,2.1,1.8,1.4,1.1]

storage    = initial_storage
day_index  = 0
total_spill = 0.0

print(f"{'Day':>4} {'Inflow':>7} {'Outflow':>8} {'Net':>6} {'Storage':>8}  Note")
print("─"*48)

while storage < capacity_Mm3 and day_index < len(inflow_Mm3day):
    Qin     = inflow_Mm3day[day_index]
    net     = Qin - outflow_Mm3day
    storage += net
    note    = ""
    spill   = 0.0
    if storage >= capacity_Mm3:
        spill       = storage - capacity_Mm3
        storage     = capacity_Mm3
        total_spill += spill
        note        = f"SPILL {spill:.2f}" if spill > 0 else "FULL"
    day_index += 1
    print(f"{day_index:>4} {Qin:>7.2f} {outflow_Mm3day:>8.2f} {net:>6.2f} {storage:>8.2f}  {note}")

if storage >= capacity_Mm3:
    print(f"
Reservoir FULL on Day {day_index}. Total spill: {total_spill:.2f} Mm3")
else:
    print(f"
Monsoon ended. Final storage: {storage:.2f} Mm3 ({storage/capacity_Mm3*100:.1f}% full)")

### 🔁 Try this

Change `outflow_Mm3day = 0.8` to `outflow_Mm3day = 3.0` (increased release for flood control).

- Does the reservoir fill?
- What is the final storage?
- What does this tell a reservoir operator about the trade-off between irrigation supply and flood control?

---
## Code Block 2 — Back-calculating Manning's n by Iteration

### What this code does

We find Manning's roughness coefficient n for a pipe where the observed velocity is known, by iteratively adjusting n_guess until the computed velocity matches the observed value to within a tight tolerance.

### Why each step is taken

**Start from analytical solution check:**
We compute the exact n algebraically first — this tells us what value the iteration should converge to, providing a verification target.

**`n_guess = n_guess * (V_calc / V_observed)`:**
The update rule. If V_calc > V_observed, then n_guess is too small (low roughness → too fast). Multiplying by (V_calc/V_observed) increases n_guess proportionally.

**`while abs(error) >= tol and iterations < 50`:**
Two exit conditions: converged (error is small enough) OR safety limit (max 50 iterations). The safety limit prevents an infinite loop if the method fails to converge.

### Algorithm

```
1. V_obs=1.45, R=0.38, S=0.002
   n_exact = (1/V_obs)*R^(2/3)*S^0.5  → exact answer

2. n_guess=0.020, tol=1e-6

3. while |error| >= tol AND iter < 50:
     V_calc = (1/n_guess)*R^(2/3)*S^0.5
     error  = V_obs - V_calc
     n_guess = n_guess*(V_calc/V_obs)
     iter   += 1

4. Print final n with number of iterations
```

### Expected output

```
Exact n = 0.016413
 Iter   n_guess    V_calc     Error
    0   0.020000    1.1899   0.260100
    1   0.016413    1.4500   0.000002
    2   0.016413    1.4500   0.000000
Converged: n = 0.016413 in 2 iterations
```

In [ ]:
import math

V_observed = 1.45   # measured velocity, m/s
R          = 0.38   # hydraulic radius, m
S          = 0.002  # measured slope

# Exact answer for verification
n_exact = (1/V_observed) * R**(2/3) * S**0.5
print(f"Exact n = {n_exact:.6f}")
print()

# Iterative solution
n_guess    = 0.020
tol        = 1e-6
iterations = 0

print(f"{'Iter':>5} {'n_guess':>10} {'V_calc':>10} {'Error':>10}")
while True:
    V_calc = (1/n_guess) * R**(2/3) * S**0.5
    error  = V_observed - V_calc
    print(f"{iterations:>5} {n_guess:>10.6f} {V_calc:>10.4f} {error:>10.6f}")
    if abs(error) < tol:
        break
    n_guess    = n_guess * (V_calc / V_observed)
    iterations += 1
    if iterations >= 50:
        print("Max iterations reached — did not converge")
        break

print(f"Converged: n = {n_guess:.6f} in {iterations} iterations")

### 🔁 Try this

Change the starting guess to `n_guess = 0.050` (much worse initial estimate).

- Does it still converge to the same answer?
- Does it take more iterations?
- Try `n_guess = 0.005` — what happens?

---
## Code Block 3 — Hardy-Cross Pipe Network

### What this code does

We solve a 2-pipe parallel network using the Hardy-Cross method — iteratively adjusting flow until head losses balance to within a tolerance.

### Why each step is taken

**`def head_loss(Q, L, D)`:**
A helper function avoids duplicating the Darcy-Weisbach formula inside the loop. This is good practice when the same calculation is performed multiple times.

**`dQ = -(hf1 - hf2) / (2 * (hf1/Q1 + hf2/Q2))`:**
The Hardy-Cross correction formula. The numerator is the head loss imbalance (should be zero at equilibrium). The denominator is the sum of dh_f/dQ for each pipe. The correction is subtracted from Q1 and added to Q2.

**`while abs(hf1 - hf2) >= tol`:**
Continue until the head loss imbalance is smaller than the tolerance. For a well-behaved 2-pipe network this typically converges in 5-10 iterations.

### Algorithm

```
1. Two parallel pipes:
   Pipe 1: L=200m, D=200mm
   Pipe 2: L=300m, D=250mm
   Q_total = 0.05 m³/s

2. Initial split: Q1=Q_total*(D1²/(D1²+D2²)), Q2=Q_total-Q1

3. while |hf1-hf2| >= 0.0001:
     hf1, hf2 from D-W formula
     imbalance = hf1 - hf2
     dQ = -imbalance/(2*(hf1/Q1+hf2/Q2))
     Q1 += dQ; Q2 -= dQ

4. Print converged Q1, Q2, hf
```

### Expected output

```
 Iter   Q1(L/s)  Q2(L/s)   hf1(m)   hf2(m)  Imbal
    0    18.18    31.82    3.283    3.135    0.148
    1    18.27    31.73    3.294    3.125    0.169
...
CONVERGED in X iterations: Q1=18.X L/s, Q2=31.X L/s, hf=3.XX m
```

In [ ]:
import math

f=0.018; g=9.81; Q_total=0.05

def head_loss(Q, L, D):
    A = math.pi*(D/2)**2
    V = Q/A
    return f*(L/D)*(V**2/(2*g))

L1=200; D1=0.200; L2=300; D2=0.250
# Initial flow split proportional to D²
Q1 = Q_total * D1**2 / (D1**2 + D2**2)
Q2 = Q_total - Q1

tol        = 0.0001
iterations = 0
print(f"{'Iter':>5} {'Q1(L/s)':>9} {'Q2(L/s)':>9} {'hf1(m)':>8} {'hf2(m)':>8} {'Imbal':>8}")

while True:
    hf1   = head_loss(Q1, L1, D1)
    hf2   = head_loss(Q2, L2, D2)
    imbal = hf1 - hf2
    print(f"{iterations:>5} {Q1*1000:>9.2f} {Q2*1000:>9.2f} {hf1:>8.3f} {hf2:>8.3f} {imbal:>8.4f}")
    if abs(imbal) < tol:
        break
    dQ = -imbal / (2*(hf1/Q1 + hf2/Q2))
    Q1 += dQ; Q2 -= dQ
    iterations += 1
    if iterations >= 50:
        print("Max iterations reached")
        break

print(f"
CONVERGED in {iterations} iterations: Q1={Q1*1000:.2f} L/s, Q2={Q2*1000:.2f} L/s, hf={hf1:.3f} m")

### 🔁 Try this

Change both pipes to the same diameter (D1=D2=0.225m).

- Does the flow split evenly (Q1=Q2=25 L/s)?
- Are the head losses equal?
- How many iterations does convergence take?

---
## Session Summary — while Loops

| Concept | Syntax | Notes |
|---|---|---|
| Basic while | `while condition:` | Exits when condition False |
| Infinite loop + break | `while True: ... break` | Manual exit on convergence |
| Safety exit | `if iter >= 50: break` | Prevent infinite loops |
| AND condition | `while c1 and c2:` | Both must hold |
| Index increment | `day_index += 1` | Must be done manually |
| Helper function | `def hf(Q,L,D): ...` | Avoid duplicate formulas |

---
## Day 15 Assignment

Detention pond routing with hourly inflow. Stop when pond is full (capacity=8000 m³) or inflow ends.

### ▶ Assignment cell

In [ ]:
inflow_m3hr = [200,450,890,1240,980,720,540,380,250,180,120,80]
capacity   = 8000.0
storage    = 500.0
outflow    = 300.0
warning_80 = capacity * 0.8
hour       = 0

print(f"{'Hr':>4} {'Inflow':>8} {'Out':>6} {'Storage':>10} {'%Full':>7} {'Note'}")
print("-"*50)

while hour < len(inflow_m3hr):
    Qin     = inflow_m3hr[hour]
    net     = Qin - outflow
    storage += net
    overflow = 0
    note     = ""

    if storage >= capacity:
        overflow = storage - capacity
        storage  = capacity
        note     = f"OVERFLOW {overflow:.0f}"

    pct = storage/capacity*100
    print(f"{hour+1:>4} {Qin:>8} {outflow:>6.0f} {storage:>10.1f} {pct:>6.1f}%  {note}")

    if storage >= capacity:
        print(f"
Pond FULL at hour {hour+1}.")
        break
    hour += 1
else:
    print(f"
Inflow ended. Final storage: {storage:.0f} m3 ({storage/capacity*100:.1f}%)")

---
- [ ] Run all cells — verify outputs match expected outputs
- [ ] Complete the assignment cell
- [ ] Upload: `Unit2_LoopsDecisions/CE541E08_U2_Day15.ipynb`
- [ ] Commit: `Day 15 assignment completed`

*CE541E08 · Civil Engineering · Christ University · 2026-27 · Dr. Arpan Pradhan*